In [1]:
import numpy as np

from model import DISMAL

# Reproduction of Casper (2014) with density evolution
# https://iopscience.iop.org/article/10.1088/0029-5515/54/1/013005/meta

# Set timesteps
rampup_times = np.linspace(5.0, 80.0, 10)
flattop_times = np.linspace(100.0, 450.0, 10)
rampdown_times = np.linspace(500.0, 600.0, 10)
times = np.r_[rampup_times, flattop_times, rampdown_times]

# Load gEQDSK
# g_arr_rampup = [f'eqdsk/iter_i={i}.eqdsk' for i in range(10)]
g_arr_rampup = [f'eqdsk/iter_i={i}.eqdsk' for i in range(10)]
g_arr_flattop = ['eqdsk/Hmode.eqdsk'] * 10
g_arr_rampdown = g_arr_rampup[::-1]
g_arr = np.r_[g_arr_rampup, g_arr_flattop, g_arr_rampdown]

# Set current
ip = {0: 3.0E6, 5: 3.0E6, 80: 15.0E6, 500: 15.0E6, 590: 4.0E6, 600: 4.0E6}

# Set heating
powers = {0: 0, 24: 0, 25: 10.0E6, 79: 10.0E6, 80: 52.0E6, 124: 52.0E6, 125: 40.0E6, 500: 40.0E6, 524: 40.0E6, 525: 35.0E6, 549: 35.0E6, 550: 30.0E6}
nbi_powers = {k: 0.5 * v for k, v in powers.items()}
eccd_powers = {k: 0.5 * v for k, v in powers.items()}

# Set pedestals
T_i_ped = {0: 0.146, 80: 0.146, 85: 3.69, 500: 3.69, 505: 0.146}
T_e_ped = {0: 0.220, 80: 0.220, 85: 3.69, 500: 3.69, 505: 0.220}
# n_e_ped = {0: 1.821E19, 79: 1.821E19, 80: 7.482E19} # old
n_e_ped = {0: 1.821E19, 79: 1.821E19, 80: 7.482E19, 500: 7.482E19, 505: 1.821E19} # new 2026-01-20 from j lhote


# Set density profiles
def get_data(fname, mult):
    f = open(fname)
    raw = f.readlines()
    dict = {}
    for line in raw:
        x, y = line.split(',')
        x = float(x.strip())
        y = float(y.strip()) * mult
        dict[x] = y
    dict[1.0] = y

    return dict

# Set boundary conditions
# ne_right_bc = 1.0E18
ne_right_bc = {0: 0.157E20, 79: 0.157E20, 80: 0.414E20}
Te_right_bc = 0.01
Ti_right_bc = 0.01

# Run sim
t_res = np.arange(0.0, 600.0, 20.0)

mysim = DISMAL(0, 600, times, g_arr, times=t_res, dt=1.0, last_surface_factor=0.9)
mysim.initialize_gs('ITER_mesh.h5', vsc='VS')
coil_names = ['CS3U', 'CS2U', 'CS1U', 'CS1L', 'CS2L', 'CS3L', 'PF1', 'PF2', 'PF3', 'PF4', 'PF5', 'PF6']
target_currents = {coil: 0.0 for coil in coil_names}
mysim.set_coil_reg(targets=target_currents, strict_limit=1.0E8)

mysim.set_Ip(ip)
mysim.set_Zeff(1.8)
mysim.set_heating(nbi=nbi_powers, nbi_loc=0.25, eccd=eccd_powers, eccd_loc=0.35)
mysim.set_right_bc(Te_right_bc=Te_right_bc, Ti_right_bc=Ti_right_bc, ne_right_bc=ne_right_bc)
mysim.set_pedestal(T_i_ped=T_i_ped, T_e_ped=T_e_ped, n_e_ped=n_e_ped)
mysim.set_nbar({0: 0.326E20, 80: .905E20})




mysim.fly(save_states=True, graph=False, gif=True, max_step = 5)

#----------------------------------------------
Open FUSION Toolkit Initialized
Development branch:   main
Revision id:          9d28a73
Parallelization Info:
  Not compiled with MPI
  # of OpenMP threads =    2
Fortran input file    = /var/folders/9p/90ydvncx0zb8sqr976fmrfxr0000gn/T/oft_9244/oftpyin
XML input file        = none
Integer Precisions    =    4   8
Float Precisions      =    4   8  16
Complex Precisions    =    4   8
LA backend            = native
#----------------------------------------------


**** Loading OFT surface mesh

**** Generating surface grid level  1
  Generating boundary domain linkage
  Mesh statistics:
    Area         =  2.859E+02
    # of points  =    4757
    # of edges   =   14156
    # of cells   =    9400
    # of boundary points =     112
    # of boundary edges  =     112
    # of boundary cells  =     112
  Resolution statistics:
    hmin =  9.924E-03
    hrms =  2.826E-01
    hmax =  8.466E-01
  Surface grounded at vertex     870


**** Creating 

TypeError: DISMAL.fly() got an unexpected keyword argument 'gif'

In [ ]:
# import shutil
# import os

# # def fly(mysim, convergence_threshold=-1.0, save_states=False, graph=False, max_step=5, out='res.json'):
# out='res.json'
# convergence_threshold = -1.0
# save_states = False
# graph = False
# max_step = 2


# del_tmp = input('Delete temporary storage? [y/n] ')
# if del_tmp != 'y':
#     quit()
# with open('convergence_history.txt', 'w'):
#     pass
# shutil.rmtree('./tmp')
# os.mkdir('./tmp')

# mysim._fname_out = out

# # if graph:
# #     for var in ['ffp_prof', 'pp_prof']:
# #         fig, ax = plt.subplots(1, len(mysim._eqtimes))
# #         fig.suptitle(var)
# #         for i, _ in enumerate(mysim._eqtimes):
# #             ax[i].plot(mysim._state[var][i]['x'], mysim._state[var][i]['y'])
# #         plt.show()           

# err = convergence_threshold + 1.0
# step = 0
# cflux_prev = 0.0

# while err > convergence_threshold and step < max_step:
#     cflux, data_tree = mysim._run_transport(step, graph=graph)
#     if save_states:
#         mysim.save_state('tmp/ts_state{}.json'.format(step))
#     step += 1

#     cflux_gs = mysim._run_gs(step, graph=graph)
#     if save_states:
#         mysim.save_state('tmp/gs_state{}.json'.format(step-1))

#     mysim.save_res()

#     with open('convergence_history.txt', 'a') as f:
#         print("GS CF = {}".format(cflux_gs), file=f)
#         print("TS CF = {}".format(cflux), file=f)

#     err = np.abs(cflux - cflux_prev) / cflux_prev
#     cflux_prev = cflux

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# # ==== EXPLORING THE STRUCTURE ====

# # Print full tree structure
# print(data_tree)

# # Get all groups
# print("\nAvailable groups:")
# print(list(data_tree.groups))

# # Access specific groups (use actual names from tree)
# numerics = data_tree['numerics']
# profiles = data_tree['profiles']
# scalars = data_tree['scalars']

# # See what's in each group
# print("\n=== Profiles variables ===")
# print(profiles.ds.data_vars)

# print("\n=== Scalars variables ===")
# print(scalars.ds.data_vars)

# # ==== EXTRACTING DATA (like pandas) ====

# # Extract a specific profile (returns xarray DataArray)
# T_i = data_tree['profiles']['T_i']  # or profiles.ds['T_i']
# T_e = data_tree['profiles']['T_e']
# n_e = data_tree['profiles']['n_e']

# # Extract a scalar timeseries
# Ip = data_tree['scalars']['Ip']
# W_thermal_i = data_tree['scalars']['W_thermal_i']

# # Convert to numpy array (like .values in pandas)
# T_i_array = T_i.values  # Shape: (time, rho_norm)
# Ip_array = Ip.values    # Shape: (time,)

# # Get coordinates
# time_coords = T_i.coords['time'].values
# rho_coords = T_i.coords['rho_norm'].values

# # ==== SLICING (like pandas) ====

# # Select specific time
# T_i_at_t0 = T_i.sel(time=0.0, method='nearest')

# # Select time range
# T_i_range = T_i.sel(time=slice(100, 200))

# # Select specific rho
# T_i_at_rho05 = T_i.sel(rho_norm=0.5, method='nearest')

# # Multi-dimensional indexing
# T_i_point = T_i.sel(time=100, rho_norm=0.5, method='nearest')

# # ==== QUICK PLOTTING ====

# # Plot scalar vs time
# plt.figure(figsize=(12, 4))
# plt.subplot(131)
# Ip.plot()
# plt.title('Plasma Current')

# # Plot profile at specific time
# plt.subplot(132)
# T_i.sel(time=100, method='nearest').plot()
# plt.title('Ion Temperature at t=100')

# # Plot 2D heatmap
# plt.subplot(133)
# T_i.plot(x='time', y='rho_norm', cmap='hot')
# plt.title('T_i Evolution')
# plt.tight_layout()
# plt.show()

# # ==== OPERATIONS (like pandas) ====

# # Time average
# T_i_mean = T_i.mean(dim='time')

# # Maximum over time
# T_i_max = T_i.max(dim='time')

# # Integration over rho
# integrated = (T_i * n_e).integrate('rho_norm')

# # Compute derived quantities
# stored_energy = W_thermal_i + data_tree['scalars']['W_thermal_e']

# # ==== INSPECT SPECIFIC TIMES ====
# print(f"\nT_i at center (rho=0) over time:")
# print(T_i.sel(rho_norm=0.0, method='nearest').values[:10])

# print(f"\nPlasma current evolution:")
# print(Ip.values[:10])